In [2]:
"""
Ice crystal oscillation animation.

The crystal experiences a sinusoidally varying temperature:
    T(t) = T0 + dT * sin(2π t / tau)

Mass evolves via the Mason (1971) spherical growth equation.
The crystal is drawn as a scatter point whose area scales with r².
A side panel shows the real-time mass and temperature traces.

Usage:
    python ice_crystal_animation.py

Output:
    ice_crystal_oscillation.mp4  (or .gif if ffmpeg unavailable)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from matplotlib.patches import FancyArrowPatch
import warnings

# ─── Physics constants ────────────────────────────────────────────────────────
Rv      = 461.5       # J kg⁻¹ K⁻¹  specific gas constant, water vapour
Lsub    = 2.834e6     # J kg⁻¹      latent heat of sublimation
ka      = 0.024       # W m⁻¹ K⁻¹  thermal conductivity of air
Dv0     = 2.11e-5     # m² s⁻¹     diffusivity at 273 K, 1 atm
P0      = 22632.0     # Pa          approximate pressure at ~12 km
rho_ice = 917.0       # kg m⁻³
pi      = np.pi

# ─── Simulation parameters ───────────────────────────────────────────────────
T0       = 218.0      # K    mean temperature
dT       = 0.2        # K    temperature oscillation amplitude
tau_min  = 10.0       # min  oscillation period
RHI      = 1.10       # –    relative humidity w.r.t. ice at T0
r0_um    = 1.0       # μm   initial crystal radius
n_cycles = 4          # number of full oscillation cycles to animate
fps      = 30         # frames per second in output
n_frames = 300        # total animation frames

# ─── Physics helpers ─────────────────────────────────────────────────────────

def esi(T):
    """Murphy & Koop (2005) saturation vapour pressure over ice [Pa]."""
    return np.exp(9.550426 - 5723.265 / T + 3.53068 * np.log(T) - 0.00728332 * T)


def Dv(T):
    """Temperature- and pressure-corrected vapour diffusivity [m² s⁻¹]."""
    return Dv0 * (T / 273.15) ** 1.75 * (101325.0 / P0)


def dmdt(T, r_m, e_amb):
    """
    Mason (1971) mass growth rate [kg s⁻¹].
    Positive = deposition, negative = sublimation.
    """
    es_loc = esi(T)
    s      = e_amb / es_loc - 1.0
    Fk     = (Lsub / (Rv * T) - 1.0) * Lsub / (ka * T)
    Fd     = Rv * T / (Dv(T) * es_loc)
    return 4.0 * pi * r_m * s / (Fk + Fd)


# ─── Pre-simulate full trajectory ────────────────────────────────────────────

tau_s   = tau_min * 60.0
t_total = n_cycles * tau_s
e_amb   = RHI * esi(T0)

dt_sim  = tau_s / 500          # fine integration timestep
n_steps = int(t_total / dt_sim)

t_arr    = np.zeros(n_steps)
T_arr    = np.zeros(n_steps)
r_arr    = np.zeros(n_steps)   # μm
s_arr    = np.zeros(n_steps)   # supersaturation %

mass = (4.0 / 3.0) * pi * (r0_um * 1e-6) ** 3 * rho_ice

for i in range(n_steps):
    t      = i * dt_sim
    T      = T0 + dT * np.sin(2.0 * pi * t / tau_s)
    r_m    = (3.0 * mass / (4.0 * pi * rho_ice)) ** (1.0 / 3.0)
    rate   = dmdt(T, r_m, e_amb)
    mass   = max(0.0, mass + rate * dt_sim)

    t_arr[i] = t / 60.0                          # minutes
    T_arr[i] = T
    r_arr[i] = (3.0 * mass / (4.0 * pi * rho_ice)) ** (1.0 / 3.0) * 1e6
    s_arr[i] = (e_amb / esi(T) - 1.0) * 100.0

# Subsample to n_frames for animation
frame_idx = np.linspace(0, n_steps - 1, n_frames, dtype=int)
t_frames  = t_arr[frame_idx]
T_frames  = T_arr[frame_idx]
r_frames  = r_arr[frame_idx]
s_frames  = s_arr[frame_idx]

# ─── Figure layout ───────────────────────────────────────────────────────────

fig = plt.figure(figsize=(11, 6), facecolor="#0d1117")
fig.patch.set_facecolor("#0d1117")

gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[1.1, 1],
    height_ratios=[1, 1],
    hspace=0.45,
    wspace=0.38,
    left=0.07, right=0.97,
    top=0.91, bottom=0.10,
)

ax_crystal = fig.add_subplot(gs[:, 0])   # left: crystal view
ax_temp    = fig.add_subplot(gs[0, 1])   # top-right: temperature
ax_radius  = fig.add_subplot(gs[1, 1])   # bottom-right: radius

for ax in [ax_crystal, ax_temp, ax_radius]:
    ax.set_facecolor("#161b22")
    ax.tick_params(colors="#8b949e", labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")

T_lo = T0 - dT * 1.15
T_hi = T0 + dT * 1.15

# ax_crystal.set_xlim(-0.6, 0.6)
# ax_crystal.set_ylim(T_lo - 0.02, T_hi + 0.02)
# ax_crystal.set_xlabel("", color="#8b949e")
# ax_crystal.set_ylabel("Temperature  (K)", color="#8b949e", fontsize=9)
# ax_crystal.set_title("Ice Crystal Oscillation", color="#e6edf3",
#                      fontsize=11, fontweight="bold", pad=10)
# ax_crystal.set_xticks([])

# # shaded warm / cold bands
# ax_crystal.axhspan(T0, T_hi + 0.05, color="#5c1a1a", alpha=0.35, zorder=0)
# ax_crystal.axhspan(T_lo - 0.05, T0, color="#1a2e4a", alpha=0.35, zorder=0)
# ax_crystal.axhline(T0, color="#30363d", lw=0.8, ls="--", zorder=1)
# ax_crystal.text(0.52, T0 + 0.005, "T₀", color="#8b949e", fontsize=8, va="bottom")

# # labels
# ax_crystal.text(-0.55, T_hi - 0.01, "warm  ▲ sublimation",
#                 color="#f85149", fontsize=7.5, va="top", alpha=0.85)
# ax_crystal.text(-0.55, T_lo + 0.01, "cold  ▼ deposition",
#                 color="#58a6ff", fontsize=7.5, va="bottom", alpha=0.85)

# # oscillation path (faint sine)
# t_path = np.linspace(0, 2 * pi, 300)
# T_path = T0 + dT * np.sin(t_path)
# ax_crystal.plot(np.zeros(300), T_path, color="#30363d", lw=1.0,
#                 ls=":", zorder=1, alpha=0.6)

# # crystal scatter point  —  size proportional to r²
# r_max   = r_frames.max()
# MAX_PT  = 800     # max marker area (pt²)
# MIN_PT  = 10

# def r_to_size(r):
#     return MIN_PT + (MAX_PT - MIN_PT) * (r / r_max) ** 2

# scat = ax_crystal.scatter(
#     [0], [T_frames[0]],
#     s=r_to_size(r_frames[0]),
#     c=[s_frames[0]],
#     cmap="RdBu_r",
#     vmin=-2, vmax=2,
#     edgecolors="#e6edf3",
#     linewidths=0.6,
#     zorder=5,
# )

# # crystal radius annotation
# radius_text = ax_crystal.text(
#     0.0, T_lo - 0.005,
#     f"r = {r_frames[0]:.2f} μm",
#     color="#e6edf3", fontsize=8.5,
#     ha="center", va="top",
#     transform=ax_crystal.get_xaxis_transform(),
#     zorder=6,
# )

# # time annotation
# time_text = ax_crystal.text(
#     0.97, 0.97, f"t = {t_frames[0]:.1f} min",
#     transform=ax_crystal.transAxes,
#     color="#8b949e", fontsize=8,
#     ha="right", va="top",
# )

# # colorbar for supersaturation
# cbar = fig.colorbar(scat, ax=ax_crystal, orientation="horizontal",
#                     fraction=0.046, pad=0.12, aspect=25)
# cbar.set_label("Supersaturation (%)", color="#8b949e", fontsize=8)
# cbar.ax.tick_params(colors="#8b949e", labelsize=7)
# cbar.outline.set_edgecolor("#30363d")

# ── Temperature trace ─────────────────────────────────────────────────────────
ax_temp.set_xlim(t_frames[0], t_frames[-1])
ax_temp.set_ylim(T_lo - 0.02, T_hi + 0.02)
ax_temp.set_ylabel("T  (K)", color="#8b949e", fontsize=8)
ax_temp.set_xlabel("Time  (min)", color="#8b949e", fontsize=8)
ax_temp.set_title("Temperature", color="#c9d1d9", fontsize=9, pad=6)
ax_temp.plot(t_arr, T_arr, color="#30363d", lw=0.8, zorder=1)
ax_temp.axhline(T0, color="#484f58", lw=0.7, ls="--")

line_T,    = ax_temp.plot([], [], color="#f78166", lw=1.6, zorder=3)
dot_T      = ax_temp.scatter([], [], s=40, color="#f78166", zorder=4)

# ── Radius trace ──────────────────────────────────────────────────────────────
ax_radius.set_xlim(t_frames[0], t_frames[-1])
ax_radius.set_ylim(max(0, r_frames.min() * 0.95), r_frames.max() * 1.05)
ax_radius.set_ylabel("r  (μm)", color="#8b949e", fontsize=8)
ax_radius.set_xlabel("Time  (min)", color="#8b949e", fontsize=8)
ax_radius.set_title("Crystal radius", color="#c9d1d9", fontsize=9, pad=6)
ax_radius.plot(t_arr, r_arr, color="#30363d", lw=0.8, zorder=1)
ax_radius.axhline(r0_um, color="#484f58", lw=0.7, ls="--")
ax_radius.text(t_frames[-1] * 0.98, r0_um * 1.002, "r₀",
               color="#484f58", fontsize=7, ha="right", va="bottom")

line_R,    = ax_radius.plot([], [], color="#79c0ff", lw=1.6, zorder=3)
dot_R      = ax_radius.scatter([], [], s=40, color="#79c0ff", zorder=4)

# ─── Animation update ─────────────────────────────────────────────────────────

def update(frame):
    i   = frame_idx[frame]
    t   = t_frames[frame]
    T   = T_frames[frame]
    r   = r_frames[frame]
    s   = s_frames[frame]

    # traces (draw up to current frame)
    mask = t_frames <= t
    line_T.set_data(t_frames[mask], T_frames[mask])
    dot_T.set_offsets([[t, T]])

    line_R.set_data(t_frames[mask], r_frames[mask])
    dot_R.set_offsets([[t, r]])

    return line_T, dot_T, line_R, dot_R

# ─── Build & save ─────────────────────────────────────────────────────────────

anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / fps, blit=True)

out_mp4 = "ice_crystal_oscillation.mp4"
out_gif = "ice_crystal_oscillation.gif"

try:
    writer = FFMpegWriter(fps=fps, bitrate=1800,
                          extra_args=["-vcodec", "libx264", "-pix_fmt", "yuv420p"])
    anim.save(out_mp4, writer=writer, dpi=150)
    print(f"Saved → {out_mp4}")
except Exception as e:
    warnings.warn(f"ffmpeg unavailable ({e}), falling back to GIF.")
    writer = PillowWriter(fps=fps)
    anim.save(out_gif, writer=writer, dpi=120)
    print(f"Saved → {out_gif}")

plt.close(fig)


Saved → ice_crystal_oscillation.mp4
